# ARC-AGI-2 — Recursive Transformer and submission generator

This notebook reads ARC-AGI-2 challenge JSON, trains a small recursive transformer on leave-one-out training demonstrations, optionally adapts on each task's visible examples, and writes `submission.json` in the competition format. It runs without Unsloth, model downloads, or internet access. This is a baseline implementation; an untrained or lightly trained model is not expected to solve arbitrary ARC tasks.

In [ ]:
from pathlib import Path
import json, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_DIR = Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-2')
if not DATA_DIR.exists():
    found = list(Path('/kaggle/input').glob('**/arc-agi_training_challenges.json'))
    if found: DATA_DIR = found[0].parent
TRAIN_PATH = DATA_DIR / 'arc-agi_training_challenges.json'
TEST_PATH = DATA_DIR / 'arc-agi_test_challenges.json'
EVAL_PATH = DATA_DIR / 'arc-agi_evaluation_challenges.json'
OUTPUT_PATH = Path('/kaggle/working/submission.json') if Path('/kaggle/working').exists() else Path('submission.json')
ROW, EOS, SEP, IN, OUT, QUERY, ANSWER, BOS, PAD = range(10, 19)
VOCAB_SIZE = 19

def read_json(path):
    with open(path, encoding='utf-8') as f: return json.load(f)

if not TRAIN_PATH.exists(): raise FileNotFoundError(f'Missing {TRAIN_PATH}')
train_tasks = read_json(TRAIN_PATH)
if TEST_PATH.exists():
    inference_tasks = read_json(TEST_PATH)
elif EVAL_PATH.exists():
    inference_tasks = read_json(EVAL_PATH)
else:
    raise FileNotFoundError('Neither test nor evaluation challenge JSON was found')
print('device:', DEVICE, '| training tasks:', len(train_tasks), '| inference tasks:', len(inference_tasks))

In [ ]:
def grid_tokens(grid):
    a = np.asarray(grid)
    if a.ndim != 2 or not (1 <= a.shape[0] <= 30 and 1 <= a.shape[1] <= 30): raise ValueError(f'bad grid shape: {a.shape}')
    if np.any((a < 0) | (a > 9)): raise ValueError('grid colors must be integers 0..9')
    out = []
    for row in a.tolist(): out.extend(int(v) for v in row); out.append(ROW)
    return out + [EOS]

def parse_grid(tokens):
    rows, row = [], []
    for token in tokens:
        token = int(token)
        if token == EOS: break
        if 0 <= token <= 9: row.append(token)
        elif token == ROW and row: rows.append(row); row = []
    if row: rows.append(row)
    if not rows or len(rows) > 30 or not rows[0] or len(rows[0]) > 30: return None
    if any(len(r) != len(rows[0]) for r in rows): return None
    return rows

def make_prompt(task, query_grid, heldout=None):
    tokens = []
    for i, pair in enumerate(task.get('train', [])):
        if i == heldout: continue
        tokens += [IN] + grid_tokens(pair['input']) + [OUT] + grid_tokens(pair['output']) + [SEP]
    return tokens + [QUERY] + grid_tokens(query_grid) + [ANSWER]

## Recursive transformer

The shared Transformer encoder is applied recursively. Input, output-draft, and latent tokens have separate type and position embeddings. The output sequence is decoded autoregressively so grid dimensions can vary.

In [ ]:
class RecursiveARCTransformer(nn.Module):
    def __init__(self, d_model=128, nhead=4, layers=2, z_slots=4, max_positions=2048, recursion_steps=2):
        super().__init__()
        self.z_slots, self.recursion_steps = z_slots, recursion_steps
        self.token = nn.Embedding(VOCAB_SIZE, d_model, padding_idx=PAD)
        self.kind = nn.Embedding(3, d_model)
        self.position = nn.Embedding(max_positions, d_model)
        block = nn.TransformerEncoderLayer(d_model, nhead, 4*d_model, batch_first=True, norm_first=True, activation='gelu')
        self.core = nn.TransformerEncoder(block, layers, enable_nested_tensor=False)
        self.z_init = nn.Parameter(torch.randn(1, z_slots, d_model) * 0.02)
        self.update_y = nn.Linear(d_model, d_model); self.update_z = nn.Linear(d_model, d_model)
        self.norm = nn.LayerNorm(d_model); self.head = nn.Linear(d_model, VOCAB_SIZE)
        self.halt = nn.Linear(d_model, 1)

    def forward(self, x, y, recursion_steps=None):
        b, nx = x.shape; ny = y.shape[1]; nz = self.z_slots; total = nx + ny + nz
        if total > self.position.num_embeddings: raise ValueError('sequence exceeds position embedding limit')
        dev = x.device; xp = torch.arange(nx, device=dev); yp = torch.arange(ny, device=dev); zp = torch.arange(nz, device=dev)
        xs = self.token(x) + self.kind.weight[0] + self.position(xp)[None]
        ys = self.token(y) + self.kind.weight[1] + self.position(yp + nx)[None]
        zs = self.z_init.expand(b, -1, -1) + self.kind.weight[2] + self.position(zp + nx + ny)[None]
        mask = torch.zeros((total, total), dtype=torch.bool, device=dev)
        mask[:nx, nx:] = True
        mask[nx:nx+ny, nx:nx+ny] = torch.triu(torch.ones((ny, ny), dtype=torch.bool, device=dev), diagonal=1)
        mask[nx:nx+ny, nx+ny:] = True; mask[nx+ny:, nx:nx+ny] = True
        steps = self.recursion_steps if recursion_steps is None else recursion_steps
        for step in range(steps):
            ctx = torch.enable_grad() if step == steps-1 else torch.no_grad()
            with ctx:
                h = self.core(torch.cat([xs, ys, zs], dim=1), mask=mask)
                yh = h[:, nx:nx+ny]; zh = h[:, nx+ny:]
                logits = self.head(self.norm(yh))
                ys = ys + self.update_y(yh); zs = zs + self.update_z(zh)
            if step < steps-1: ys, zs = ys.detach(), zs.detach()
        return logits, self.halt(zs.mean(dim=1).detach()).squeeze(-1)

model = RecursiveARCTransformer().to(DEVICE)
print('parameters:', sum(p.numel() for p in model.parameters()))

## Train and adapt

Global examples hold out one visible demonstration pair and train the model to predict its output from the remaining pairs. Optional per-task adaptation uses only that task's demonstrations.

In [ ]:
GLOBAL_TRAIN = True
TASK_ADAPT = True
GLOBAL_MAX_EXAMPLES = 1200  # set to 0 for all examples
GLOBAL_EPOCHS = 1
ADAPT_EPOCHS = 2
MAX_TARGET_TOKENS = 932

def build_examples(tasks, cap=0):
    examples = []
    for task in tasks.values():
        for i, pair in enumerate(task.get('train', [])):
            try: examples.append((make_prompt(task, pair['input'], i), grid_tokens(pair['output'])))
            except (ValueError, KeyError): continue
            if cap and len(examples) >= cap: return examples
    return examples

def fit_examples(net, examples, epochs=1, lr=2e-4):
    if not examples: return
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=0.01); net.train()
    for epoch in range(epochs):
        random.shuffle(examples); losses = []
        for prompt, target in examples:
            target = target[:MAX_TARGET_TOKENS]
            x = torch.tensor([prompt], dtype=torch.long, device=DEVICE)
            y = torch.tensor([[BOS] + target[:-1]], dtype=torch.long, device=DEVICE)
            labels = torch.tensor([target], dtype=torch.long, device=DEVICE)
            opt.zero_grad(set_to_none=True); logits, halt = net(x, y)
            loss = F.cross_entropy(logits.reshape(-1, VOCAB_SIZE), labels.reshape(-1))
            loss = loss + 0.05 * F.binary_cross_entropy_with_logits(halt, torch.ones_like(halt))
            loss.backward(); torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0); opt.step()
            losses.append(float(loss.detach()))
        print(f'epoch {epoch+1}/{epochs}: loss={np.mean(losses):.4f}')
    net.eval()

if GLOBAL_TRAIN:
    examples = build_examples(train_tasks, GLOBAL_MAX_EXAMPLES)
    print('global examples:', len(examples))
    fit_examples(model, examples, GLOBAL_EPOCHS)
else: model.eval()

def adapt_task(task):
    if TASK_ADAPT:
        examples = build_examples({'task': task})
        fit_examples(model, examples, ADAPT_EPOCHS, lr=5e-5)

In [ ]:
@torch.no_grad()
def decode_grid(net, prompt, max_tokens=MAX_TARGET_TOKENS, variant=0):
    net.eval(); generated = []
    for step in range(max_tokens):
        x = torch.tensor([prompt], dtype=torch.long, device=DEVICE)
        y = torch.tensor([[BOS] + generated], dtype=torch.long, device=DEVICE)
        if len(prompt) + len(generated) + 1 + net.z_slots > net.position.num_embeddings: break
        logits, _ = net(x, y, recursion_steps=1); scores = logits[0, -1].clone()
        scores[[PAD, BOS, IN, OUT, QUERY, ANSWER, SEP]] = -1e9
        if variant and step == 0: scores[EOS] = -1e9
        token = int(scores.argmax().item()); generated.append(token)
        if token == EOS: break
    return parse_grid(generated)

def valid_grid(grid):
    return (isinstance(grid, list) and 1 <= len(grid) <= 30 and isinstance(grid[0], list) and 1 <= len(grid[0]) <= 30
            and all(isinstance(row, list) and len(row) == len(grid[0]) and all(isinstance(v, int) and 0 <= v <= 9 for v in row) for row in grid))

def create_submission(tasks):
    output = {}
    for task_id, task in tasks.items():
        adapt_task(task); answers = []
        for test in task.get('test', []):
            prompt = make_prompt(task, test['input'])
            guesses = [decode_grid(model, prompt, variant=0), decode_grid(model, prompt, variant=1)]
            guesses = [g for g in guesses if valid_grid(g)]
            if not guesses: guesses = [[[0]]]  # valid schema fallback when decoding is malformed
            if len(guesses) == 1: guesses.append(guesses[0])
            answers.append({'attempt_1': guesses[0], 'attempt_2': guesses[1]})
        output[task_id] = answers
    return output

submission = create_submission(inference_tasks)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_PATH, 'w', encoding='utf-8') as f: json.dump(submission, f)
print('wrote', OUTPUT_PATH, '| tasks:', len(submission))

In [ ]:
# Validate exact competition keys, per-test cardinality, and grid shape/value constraints.
assert isinstance(submission, dict)
for task_id, answers in submission.items():
    assert len(answers) == len(inference_tasks[task_id].get('test', []))
    for answer in answers:
        assert set(answer) == {'attempt_1', 'attempt_2'}
        assert valid_grid(answer['attempt_1']) and valid_grid(answer['attempt_2'])
with open(OUTPUT_PATH, encoding='utf-8') as f: assert json.load(f) == submission
print('submission.json validated:', OUTPUT_PATH)